# UMUD segment-then-measure (Kaggle GPU)

Trains the aponeurosis and fascicle U-Nets, predicts masks on the 309 test images, measures geometry (PCA-fit pennation + length-weighted median, TTA-denoised masks), applies the **per-family tick/ruler scale router** (`scale_ticks.py`) for muscle thickness on ~254/309 images, derives fascicle length from the MT/sin(PA) identity, and writes `submission_segmentation.csv`.

**How to run (no local setup needed):**
1. Open this notebook on Kaggle (New Notebook, then File > Import Notebook, upload this `.ipynb`).
2. Right panel > **Add Input** > add the competition *UMUD Challenge: Muscle Architecture in Ultrasound Data*.
3. Right panel > Settings: **Accelerator = GPU** (T4 or P100) and **Internet = On**.
4. **Run All**. The optional smoke cell (2 epochs) finishes in a few minutes to prove the pipeline; the full cell (12 epochs) takes roughly 30-60 minutes.
5. Cell 6 self-checks that calibration ran (per-image MT/FL, ~254 scaled). Cell 7 gives download links: the submission, the calibration debug CSV, and the trained weights (`seg_apo.pt`, `seg_fasc.pt`).

**Note:** this competition is a CSV upload. The repo already contains a validated `results/submission_local.csv` (same pipeline run on the existing saved weights, CPU). To just see the score without a 30-60 min Kaggle retrain, upload that CSV directly. Run this notebook when you want fresh weights or a reproducible end-to-end run.

In [ ]:
# Dependencies and the latest pipeline scripts (needs Internet = On)
!pip install -q segmentation-models-pytorch albumentations
!wget -q https://raw.githubusercontent.com/LacrimaeAware/kaggle-fun/main/umud-muscle-architecture/segment_then_measure.py -O segment_then_measure.py
!wget -q https://raw.githubusercontent.com/LacrimaeAware/kaggle-fun/main/umud-muscle-architecture/tick_calibration.py -O tick_calibration.py
!wget -q https://raw.githubusercontent.com/LacrimaeAware/kaggle-fun/main/umud-muscle-architecture/scale_ticks.py -O scale_ticks.py
# scale_ticks.py is the per-family scale router; segment_then_measure imports it. Fail loudly if any
# script is missing so a silent download error cannot disable calibration on the whole run.
import os
assert all(os.path.exists(f) for f in ('segment_then_measure.py', 'tick_calibration.py', 'scale_ticks.py')), \
    'a pipeline script failed to download - check Internet = On'
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

## Optional smoke run (2 epochs, a few minutes)
Proves the data paths, training loop, and submission writing before committing to the full run. Skip it if you just want the full model.

In [ ]:
!UMUD_EPOCHS=2 python segment_then_measure.py

## Full run (12 epochs)
Trains both U-Nets properly and overwrites `submission_segmentation.csv`. Also leaves `seg_apo.pt` and `seg_fasc.pt` in the working dir.

In [ ]:
!python segment_then_measure.py

In [ ]:
import pandas as pd
sub = pd.read_csv('/kaggle/working/submission_segmentation.csv')
print(sub.shape)            # expect (309, 4)
print(sub.describe())

# Sanity checks: the scale router should give per-image MT/FL, not flat constants.
assert sub.shape == (309, 4), f'expected 309x4, got {sub.shape}'
assert sub['mt_mm'].std() > 1.0, 'MT looks constant -> calibration did NOT run (scale_ticks import?)'
assert sub['fl_mm'].std() > 1.0, 'FL looks constant -> identity FL / scale did NOT run'
dbg = pd.read_csv('/kaggle/working/calibration_measurement_debug.csv')
n_scaled = int((dbg['calibration_method'] != 'none').sum())
print(f'calibrated (scaled) images: {n_scaled}/309  | methods: {dbg["calibration_method"].value_counts().to_dict()}')
assert n_scaled > 150, f'only {n_scaled} images scaled - expected ~254; check scale_ticks loaded'
print('OK: per-image MT/FL present and ~', n_scaled, 'images scaled.')

In [ ]:
# Click to download. seg_apo.pt / seg_fasc.pt are the trained weights: grab them to inspect the model locally.
import os
from IPython.display import FileLink, display
for f in ('submission_segmentation.csv', 'calibration_measurement_debug.csv', 'seg_apo.pt', 'seg_fasc.pt'):
    if os.path.exists(f):
        display(FileLink(f))